# 🌸 Flower Federated Learning Training Pipeline (FL-IoT-IDS)
## 🛡️ Industrial Edge Intrusion Detection with Flower (`flwr`), `FlowerIoTClient`, and `FlowerIoTServerStrategy`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NVKQ2022/FedLearning/blob/main/notebooks/flower_federated_pipeline.ipynb)

This interactive Jupyter Notebook implements the **Federated Learning Orchestration Layer** defined in the master's thesis proposal:
> **"Xây dựng và đánh giá prototype hệ thống phát hiện xâm nhập IoT sử dụng Federated Learning trong môi trường dữ liệu non-IID"**  
> *Nguyễn Việt Kỳ Quân, 2026*

---

### 🏛️ System Architecture: Flower Client-Server Paradigm

```mermaid
sequenceDiagram
    autonumber
    participant Server as 🌸 Flower Central Server<br/>(FlowerIoTServerStrategy)
    participant Holdout as 📊 Server Holdout Test Set<br/>(10% Val / 20% Test)
    participant C1 as 📱 Edge Client 0<br/>(FlowerIoTClient)
    participant C2 as 📱 Edge Client 1<br/>(FlowerIoTClient)
    participant Cn as 📱 Edge Client K-1<br/>(FlowerIoTClient)

    Note over Server,Holdout: Round t: Server broadcasts global weights w_t via gRPC / Ray
    Server->>C1: FitIns(w_t, config: local_epochs=2, mu=0.05)
    Server->>C2: FitIns(w_t, config: local_epochs=2, mu=0.05)
    Server->>Cn: FitIns(w_t, config: local_epochs=2, mu=0.05)

    Note over C1,Cn: Local Training on Private Non-IID Partitions (Focal Loss + Proximal Penalty)
    C1-->>Server: FitRes(w_1, n_1, {loss, acc, drift_l2, train_time})
    C2-->>Server: FitRes(w_2, n_2, {loss, acc, drift_l2, train_time})
    Cn-->>Server: FitRes(w_n, n_n, {loss, acc, drift_l2, train_time})

    Note over Server: Server Aggregation: w_{t+1} = sum(n_k / N * w_k)<br/>Communication Cost: 2 * sum(m_t * B)
    Server->>Holdout: Centralized Evaluation on Server Holdout Set
    Holdout-->>Server: Macro-F1, Minority Recall (Web-based, Brute-force), Loss
    Note over Server: Checkpoint w_{t+1} if Macro-F1 improves
```

---

### 🔑 Key Engineering & Scientific Highlights
1. **Industry-Standard Flower Framework (`flwr`):** Employs `flwr.client.NumPyClient` and custom `FlowerIoTServerStrategy` (extending `FedAvg` and `FedProx`).
2. **Server-Side Holdout Evaluation (`build_flower_server_eval_fn`):** Guarantees scientific validity by benchmarking the global model against an independent, stationary holdout set, isolating rare attacks (<1% prevalence).
3. **Thesis Section 9 Metrics Tracking:** Automatically records communication overhead ($2 	imes \sum m_t 	imes B$ in MB), parameter drift $\|w_{	ext{local}} - w_t\|_2$, and round wall-clock duration.
4. **Dual Execution Mode:** Runs seamlessly via Flower's Ray Virtual Client Engine (`start_simulation`) inside Colab/Jupyter, or as independent distributed OS processes communicating over gRPC (`start_flower_server` and `start_flower_client`).
5. **Turnkey Visualizations:** Direct integration with `src.visualization` for client label skew, convergence curves, confusion matrices, and scenario benchmarks.


---
### 🛠️ Step 1: Environment Setup & Hardware Acceleration
We verify compute resources (NVIDIA GPU or multicore CPU), sync the latest codebase from GitHub, and install all dependencies (including `flwr==1.39.0`).

> **💡 Colab Dependency Resolver Note:** Google Colab pre-installs system libraries (`bigframes`, `pyopenssl`, `google-colab`) that pin specific versions of `requests`, `rich`, and `cryptography`. When installing Flower, pip may output a dependency conflict notice. This is completely standard in Google Colab and **does not affect** Flower, PyTorch, or your model training. We add `--no-warn-conflicts` to suppress this benign warning.


In [ ]:
# 1. Hardware acceleration verification
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Compute Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"Available VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("Running on CPU. The lightweight TabularIoTMLP (~13.9K params) executes rapidly on CPU as well.")


In [ ]:
# 2. Clone or pull the latest repository codebase
import os, sys, subprocess

repo_url = "https://github.com/NVKQ2022/FedLearning.git"
repo_dir = "/content/FedLearning"

if os.path.exists("/content"):
    if not os.path.exists(repo_dir):
        print(f"Cloning {repo_url} into {repo_dir}...")
        subprocess.run(["git", "clone", repo_url, repo_dir], check=True)
    else:
        print(f"Updating repository in {repo_dir}...")
        subprocess.run(["git", "-C", repo_dir, "pull"], check=True)
    
    # Change working directory and add to sys.path
    os.chdir(repo_dir)
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
else:
    # Running locally
    if "." not in sys.path:
        sys.path.insert(0, ".")

print("Current Working Directory:", os.getcwd())


In [ ]:
# 3. Install required packages (Flower, PyTorch, Ray, Scikit-learn, etc.)
# Note: --no-warn-conflicts suppresses benign resolver notices from Colab's pre-installed bigframes/pyopenssl
!pip install -q --no-warn-conflicts "flwr>=1.13.0,<1.40.0" "ray>=2.5.0" -r requirements.txt

import flwr as fl
print(f"✅ Flower Version: {fl.__version__} (Verified & Ready)")


---
### ⚙️ Step 2: Global Hyperparameter Configuration & Reproducibility Seed
**Edit any hyperparameters here before running the simulation.** All subsequent cells consume settings directly from this single source of truth container (`ExperimentConfig`).

* **Scenarios:**
  * `Scenario E2`: IID Federated Baseline (`partition_type="iid"`, `mu=0.0`)
  * `Scenario E4`: Moderate Non-IID Dirichlet (`partition_type="dirichlet"`, `dirichlet_alpha=0.5`, `mu=0.05`)
  * `Scenario E5`: Severe Non-IID Dirichlet (`partition_type="dirichlet"`, `dirichlet_alpha=0.1`, `mu=0.1`)


In [ ]:
from src.utils.config import ExperimentConfig
from src.utils.seed import set_seed

# ==============================================================================
# 🛠️ EDIT HYPERPARAMETERS HERE (Single Source of Truth)
# ==============================================================================
CONFIG = ExperimentConfig(
    experiment_name="flower_e5_fedprox_noniid",
    seed=42,                            # Integer seed for 100% bit-level reproducibility
    
    # --- Dataset & Sampling ---
    sample_size=100000,                 # 100,000 for rapid execution, or None for full 5.11M rows
    test_size=0.20,                     # 20% holdout test set (unseen by all clients)
    val_size=0.10,                      # 10% validation set (for server-side holdout eval)
    scaler_type="robust",               # 'robust' (IQR) handles heavy-tailed network bursts
    batch_size=64,                      # Edge client mini-batch size
    
    # --- Model Architecture ---
    hidden_dims=(128, 64),              # Edge-friendly MLP without BatchNorm drift
    dropout_rate=0.2,                   # Dropout regularization
    
    # --- Loss & Imbalance Strategy ---
    loss_type="focal_loss",             # 'focal_loss' (counters extreme class imbalance)
    class_weight_strategy="balanced",   # 'balanced' (inverse frequency), 'sqrt_balanced', or 'none'
    focal_gamma=2.0,                    # Hard example focusing parameter
    
    # --- Optimization Hyperparameters ---
    optimizer_type="adamw",
    learning_rate=1e-3,                 # Client learning rate
    weight_decay=1e-4,                  # Weight decay regularization
    max_grad_norm=5.0,
    
    # --- Federated Learning & Flower Strategy ---
    num_clients=5,                      # K = 5 simulated IoT edge clients
    num_rounds=10,                      # Federated communication rounds T
    local_epochs=2,                     # Local training epochs E per round
    federated_strategy="fedprox",       # 'fedprox' (resilient to drift) or 'fedavg'
    mu=0.05,                            # FedProx proximal coefficient (mu=0 reduces to FedAvg)
    partition_type="dirichlet",         # 'dirichlet' (Non-IID) or 'iid'
    dirichlet_alpha=0.5                 # Statistical heterogeneity: 0.5 (mod), 0.1 (severe)
)

# 1. Enforce bit-level reproducibility across PyTorch, NumPy, and Python
set_seed(CONFIG.seed)

# 2. Display formatted hyperparameter configuration
CONFIG.display()


---
### 💾 Step 3: Google Drive Mounting & Dataset Acquisition
We mount Google Drive to load the real **991.00 MB** dataset (`merged_CICIOT2023_data.csv`).  
If the CSV is not found, an automatic synthetic generator will create an 8-class dataset matching the exact schema so the pipeline can dry-run immediately.


In [ ]:
# 1. Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted successfully.')
except Exception as e:
    print('Drive mount skipped or not running on Colab:', e)

import os, shutil
import pandas as pd, numpy as np

target_csv_path = CONFIG.target_csv_path if os.path.exists('/content') else 'datasets/CICIOT2023/merged_CICIOT2023_data.csv'
os.makedirs(os.path.dirname(target_csv_path), exist_ok=True)

gdrive_csv = CONFIG.gdrive_csv
if not os.path.exists(gdrive_csv):
    if os.path.exists(CONFIG.alt_gdrive_csv):
        gdrive_csv = CONFIG.alt_gdrive_csv

# 2. Acquire real dataset or fallback to synthetic dry-run
if os.path.exists(gdrive_csv):
    gdrive_size = os.path.getsize(gdrive_csv) / (1024**2)
    if not os.path.exists(target_csv_path) or os.path.getsize(target_csv_path) < 100 * 1024 * 1024:
        print(f"Copying real 991 MB dataset from Google Drive to local SSD ({gdrive_size:.2f} MB)...")
        shutil.copy(gdrive_csv, target_csv_path)
    print(f"✅ REAL CICIoT2023 dataset ready on local SSD: {os.path.getsize(target_csv_path) / (1024**2):.2f} MB")
elif os.path.exists(target_csv_path) and os.path.getsize(target_csv_path) >= 100 * 1024 * 1024:
    print(f"✅ REAL CICIoT2023 dataset already present on local SSD: {os.path.getsize(target_csv_path) / (1024**2):.2f} MB")
else:
    print(f"⚠️ Real dataset NOT found at '{gdrive_csv}'.")
    print("Generating synthetic CICIoT2023-compatible dataset (21 MB) for immediate dry-run...")
    n_samples = 30000
    classes = ['Benign', 'DDoS', 'DoS', 'Recon', 'Spoof', 'Mirai', 'Web-based', 'Brute-force']
    p = [0.18, 0.38, 0.23, 0.10, 0.05, 0.04, 0.015, 0.005]
    
    mock_data = {f'feat_{i}': np.random.exponential(scale=1.5, size=n_samples) for i in range(39)}
    mock_data['feat_0'] = np.random.pareto(a=1.5, size=n_samples) * 10.0
    mock_data['group_class_name'] = np.random.choice(classes, size=n_samples, p=p)
    
    pd.DataFrame(mock_data).to_csv(target_csv_path, index=False)
    print(f"⚠️ Synthetic dataset created ({os.path.getsize(target_csv_path)/(1024**2):.2f} MB).")


---
### 🔄 Step 4: Leak-Free Preprocessing & Class Weights
We use `load_and_preprocess_ciciot2023` to perform:
* **Stratified train/val/test split (70/10/20)** strictly before computing statistics.
* **Outlier & skew mitigation:** `RobustScaler` (median & IQR) and $\log(1+x)$ compression.
* **Class weight calculation:** Configurable strategy to penalize minority attack misclassifications.

import gc
gc.collect()


In [ ]:
from src.data.preprocess import load_and_preprocess_ciciot2023, compute_balanced_class_weights
import numpy as np

print(f"Loading and preprocessing CICIoT2023 (sample_size={CONFIG.sample_size}, scaler={CONFIG.scaler_type})...")
data = load_and_preprocess_ciciot2023(
    csv_path=target_csv_path,
    test_size=CONFIG.test_size,
    val_size=CONFIG.val_size,
    sample_size=CONFIG.sample_size,
    scaler_type=CONFIG.scaler_type,
    batch_size=CONFIG.batch_size,
    random_state=CONFIG.seed
)

class_names = data["class_names"]
minority_classes = [c for c in ["Web-based", "Brute-force"] if c in class_names]

# Compute class weights for loss function
class_weights = compute_balanced_class_weights(
    y=data["y_train"],
    num_classes=data["num_classes"],
    strategy=CONFIG.class_weight_strategy
)

print("\n--- Preprocessing Complete ---")
print(f"Train Samples: {len(data['X_train']):,}")
print(f"Val Samples:   {len(data['X_val']):,}")
print(f"Test Samples:  {len(data['X_test']):,}")
print(f"Features:      {data['input_dim']}")
print(f"Classes ({data['num_classes']}): {class_names}")
print(f"Class Weights ({CONFIG.class_weight_strategy}): {np.round(class_weights, 4)}")

import gc
gc.collect()


---
### 📊 Step 5: Edge Client Partitioning & Statistical Skew Visualization
We partition the training dataset across simulated IoT edge clients:
* **IID Partitioning (`StratifiedIIDPartitioner`):** Uniform, identical class distribution across all clients (Scenario E2).
* **Dirichlet Non-IID Partitioning (`DirichletNonIIDPartitioner`):** Real-world statistical heterogeneity where rare attacks are concentrated in specific network nodes (Scenario E4/E5).
* **Turnkey Visualization:** We plot the client class distribution using `plot_client_distribution` from `src.visualization`.


In [ ]:
from src.data.partitioner import (
    StratifiedIIDPartitioner,
    DirichletNonIIDPartitioner,
    get_client_distribution,
    create_client_dataloaders
)
from src.visualization import plot_client_distribution

# 1. Select Partitioning Scheme
if CONFIG.partition_type.lower() == "iid":
    partitioner = StratifiedIIDPartitioner(
        num_clients=CONFIG.num_clients,
        seed=CONFIG.seed
    )
else:
    partitioner = DirichletNonIIDPartitioner(
        num_clients=CONFIG.num_clients,
        alpha=CONFIG.dirichlet_alpha,
        min_samples_per_client=50,
        seed=CONFIG.seed
    )

client_partitions = partitioner.partition(
    X=data["X_train"],
    y=data["y_train"]
)

# 2. Extract Distribution DataFrame
distribution_df = get_client_distribution(
    client_partitions=client_partitions,
    y=data["y_train"],
    class_names=class_names
)

# 3. Create PyTorch DataLoaders for each client
client_loaders = create_client_dataloaders(
    X=data["X_train"],
    y=data["y_train"],
    client_partitions=client_partitions,
    batch_size=CONFIG.batch_size,
    shuffle=True
)

# 4. Visualize Client Distribution
os.makedirs("reports/figures", exist_ok=True)
fig_dist = plot_client_distribution(
    distribution=distribution_df,
    class_names=class_names,
    title=f"Edge Client Label Distribution ({CONFIG.partition_type.upper()} Partitioning)",
    save_path="reports/figures/flower_client_distribution.png",
    normalize=True,
    show=True
)

import gc
gc.collect()


---
### 🧠 Step 6: Neural Network Architecture & Server Holdout Evaluation Function
We construct the global neural network and the centralized evaluation function:
1. **`TabularIoTMLPModel`:** Edge-friendly, BatchNorm-free MLP to prevent running statistic drift across non-IID clients.
2. **`MultiClassFocalLoss`:** Focal loss with $\alpha$-class weights to counter extreme class imbalance:
   $$\mathcal{L}_{\text{Focal}} = -\sum_{c=1}^C \alpha_c (1 - p_c)^\gamma \log(p_c)$$
3. **Centralized Evaluation Function (`build_flower_server_eval_fn`):**
   * **Why evaluate on the server holdout set?** Client datasets are non-IID and partitioned; an individual client's validation set cannot reflect global defense capability. The central holdout set provides an unbiased, stationary benchmark tracking Macro-F1 and stealthy attack recalls (<1% prevalence).


In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from src.models.tabular_mlp import TabularIoTMLPModel
from src.losses.focal_loss import build_loss_function
from src.federated.flower_server import build_flower_server_eval_fn

# 1. Initialize Global Server Model
global_model = TabularIoTMLPModel(
    input_dim=data["input_dim"],
    hidden_dims=CONFIG.hidden_dims,
    num_classes=data["num_classes"],
    dropout_rate=CONFIG.dropout_rate
)
global_model.to(device)

model_size_bytes = sum(p.numel() * 4 for p in global_model.parameters())
print(f"Global Model Architecture:")
print(global_model)
print(f"Total Parameters: {sum(p.numel() for p in global_model.parameters()):,}")
print(f"Single Model Parameter Size: {model_size_bytes / 1024:.2f} KB")

# 2. Build Focal Loss Criterion
criterion = build_loss_function(
    loss_type=CONFIG.loss_type,
    class_weights=class_weights,
    gamma=CONFIG.focal_gamma,
    device=device
)

# 3. Server-side holdout evaluation callback for Flower
flower_eval_fn = build_flower_server_eval_fn(
    model=global_model,
    val_loader=data["val_loader"],
    class_names=class_names,
    criterion=criterion,
    minority_classes=minority_classes,
    device=device
)
print("Server Holdout Evaluation Callback built successfully.")


---
### 🌸 Step 7: Flower Server Strategy Configuration (`FlowerIoTServerStrategy`)
We initialize `FlowerIoTServerStrategy`, a publication-grade strategy extending Flower's `FedAvg` and `FedProx`:
* **Sample-Weighted Parameter Aggregation:** $w_{t+1} = \sum_{k=1}^K \frac{n_k}{N} w_k^t$
* **Centralized Holdout Evaluation:** Executes `flower_eval_fn` after every communication round.
* **Top Model Checkpointing:** Automatically preserves best global model weights (`flower_{strategy}_best_weights.npz`) whenever Validation Macro-F1 improves.
* **Communication Tracking:** Measures exact payload transferred ($2 \times \sum m_t \times B$ bytes).


In [ ]:
from src.federated.flower_server import FlowerIoTServerStrategy

os.makedirs("checkpoints", exist_ok=True)

strategy = FlowerIoTServerStrategy(
    strategy_name=CONFIG.federated_strategy,
    mu=CONFIG.mu if CONFIG.federated_strategy == "fedprox" else 0.0,
    fraction_fit=1.0,                    # Sample 100% of available clients per round
    min_fit_clients=CONFIG.num_clients,
    min_available_clients=CONFIG.num_clients,
    local_epochs=CONFIG.local_epochs,
    model_size_bytes=float(model_size_bytes),
    evaluate_fn=flower_eval_fn,
    checkpoint_dir="checkpoints"
)

print(f"Flower IoT Strategy Initialized:")
print(f"  Strategy Name:        {strategy.strategy_name.upper()}")
print(f"  Proximal Penalty mu:  {strategy.mu}")
print(f"  Min Fit Clients:      {strategy.min_fit_clients}")
print(f"  Local Epochs:         {strategy.local_epochs}")
print(f"  Checkpoint Dir:       {strategy.checkpoint_dir}")


---
### 📱 Step 8: Modern Flower ClientApp Factory (`FlowerIoTClient` & `ClientApp`)
We define the client factory and wrap it with Flower's modern **`ClientApp`**:
* **`FlowerIoTClient(flwr.client.NumPyClient)`:**
  * Synchronizes local model with received global weights $w_t$.
  * Executes $E$ local epochs using AdamW with gradient clipping.
  * Adds proximal penalty when $\mu > 0$: $\mathcal{L}_{\text{local}}(w) + \frac{\mu}{2} \|w - w_t\|_2^2$.
  * Computes client parameter drift norm: $\|w_{\text{local}} - w_t\|_2$.
  * Measures local wall-clock execution time and returns diagnostic metrics.
* **Modern Flower `ClientApp`:** Encapsulates `client_fn` and extracts partition configuration dynamically from Flower runtime `Context`.


In [ ]:
import flwr as fl
try:
    from flwr.client import ClientApp
except ImportError:
    from flwr.clientapp import ClientApp

from src.models.tabular_mlp import TabularIoTMLPModel
from src.federated.flower_client import FlowerIoTClient

def client_fn(context_or_cid) -> fl.client.Client:
    """
    Instantiates an ephemeral edge FlowerIoTClient for the sampled partition.
    Compatible with both modern Flower Context and legacy string cid.
    """
    if hasattr(context_or_cid, "node_config"):
        client_id = int(context_or_cid.node_config.get("partition-id", getattr(context_or_cid, "node_id", 0)))
    else:
        client_id = int(context_or_cid)
        
    local_model = TabularIoTMLPModel(
        input_dim=data["input_dim"],
        hidden_dims=CONFIG.hidden_dims,
        num_classes=data["num_classes"],
        dropout_rate=CONFIG.dropout_rate
    )
    
    # Virtual edge IoT clients train locally on CPU for robust multi-process Ray execution
    client_device = "cpu"
    
    client = FlowerIoTClient(
        client_id=client_id,
        model=local_model,
        train_loader=client_loaders[client_id],
        val_loader=data["val_loader"],
        device=client_device
    )
    return client.to_client()

# Wrap client factory into modern Flower ClientApp
client_app = ClientApp(client_fn=client_fn)

print("Modern Flower ClientApp & client_fn initialized successfully.")


---
### 🚀 Step 9: Modern Flower Simulation Execution (`flwr.simulation.run_simulation` & `ServerApp`)
We launch the multi-round federated training process using Flower's modern Simulation App Engine:
* **`ServerApp`:** Wraps server configuration and our custom `FlowerIoTServerStrategy`.
* **`flwr.simulation.run_simulation`:** Modern non-deprecated simulation runner managing virtual SuperNodes.
* **Automatic Fallback:** Gracefully falls back to classic `start_simulation` if running in older Flower environments (<1.13).


In [ ]:
import os
import time
import logging
import warnings
import flwr as fl

# 1. Memory Safety & Ray OOM Prevention (Crucial for Google Colab)
# Prevent Ray from killing worker actors due to Colab free-tier memory pressure
os.environ["RAY_memory_monitor_refresh_ms"] = "0"
os.environ["RAY_memory_usage_threshold"] = "0.99"


# 3. Detect Modern Flower App Engine
try:
    from flwr.server import ServerApp, ServerConfig
    try:
        from flwr.server import ServerAppComponents
    except ImportError:
        from flwr.serverapp import ServerAppComponents
    from flwr.simulation import run_simulation
    USE_MODERN_SIMULATION = True
except (ImportError, AttributeError):
    USE_MODERN_SIMULATION = False

print(f"Starting Flower Federated Simulation: {CONFIG.num_clients} clients, {CONFIG.num_rounds} rounds...")
print(f"Strategy: {strategy.strategy_name.upper()} | mu: {strategy.mu} | Local Epochs: {strategy.local_epochs}")
print(f"Simulation Engine: {chr(39)}Flower Next (ServerApp / ClientApp / run_simulation){chr(39)} if USE_MODERN_SIMULATION else {chr(39)}Classic Simulation (start_simulation){chr(39)}")
print("=" * 80)

sim_start_time = time.perf_counter()

# Virtual client resource allocation:
# Allocating 2 vCPUs forces Ray to train clients sequentially (1 active actor at a time),
# eliminating concurrent client RAM duplication and preventing Colab OOM crashes!
client_resources = {"num_cpus": 2, "num_gpus": 0.0}

if USE_MODERN_SIMULATION:
    def server_fn(context):
        config = ServerConfig(num_rounds=CONFIG.num_rounds)
        return ServerAppComponents(strategy=strategy, config=config)

    server_app = ServerApp(server_fn=server_fn)

    run_simulation(
        server_app=server_app,
        client_app=client_app,
        num_supernodes=CONFIG.num_clients,
        backend_config={"client_resources": client_resources}
    )
else:
    fl.simulation.start_simulation(
        client_fn=client_fn,
        num_clients=CONFIG.num_clients,
        config=fl.server.ServerConfig(num_rounds=CONFIG.num_rounds),
        strategy=strategy,
        client_resources=client_resources
    )

total_sim_time = time.perf_counter() - sim_start_time
print("=" * 80)
print(f"🎉 Flower Simulation completed in {total_sim_time:.2f} seconds ({total_sim_time / 60:.2f} minutes)!")
print(f"Best Round: {strategy.best_round} | Best Val Macro-F1: {strategy.best_macro_f1 * 100:.2f}%")


---
### 📈 Step 10: Federated Convergence & Communication Cost Diagnostics
We visualize multi-round convergence trajectories and thesis Section 9 system metrics:
1. **Validation Macro-F1 Trajectory:** Track convergence stability across rounds.
2. **Cumulative Communication Overhead:** Megabytes exchanged over gRPC ($2 \times \sum m_t \times B$).
3. **Client Parameter Drift Norm:** Average $\|w_{\text{local}} - w_t\|_2$ quantifying non-IID gradient divergence.


In [ ]:
import matplotlib.pyplot as plt
from src.visualization import plot_federated_convergence

round_history = strategy.round_history
rounds = round_history["round"]

if len(rounds) == 0:
    print("⚠️ No communication rounds were recorded in strategy.round_history.")
    print("Please check Step 9 simulation output to ensure all rounds executed successfully.")
else:
    # Align evaluation metrics (exclude round 0 if recorded before round 1)
    val_macro_f1 = round_history["val_macro_f1"]
    if len(val_macro_f1) > len(rounds):
        val_macro_f1 = val_macro_f1[-len(rounds):]

    val_loss = round_history["val_loss"]
    if len(val_loss) > len(rounds):
        val_loss = val_loss[-len(rounds):]

    # 1. Plot Validation Macro-F1 Trajectory
    fig_f1 = plot_federated_convergence(
        rounds_data={f"Flower {strategy.strategy_name.upper()} (Val Macro-F1)": val_macro_f1},
        metric_name="Macro-F1",
        title=f"Flower Federated Convergence Trajectory ({CONFIG.experiment_name})",
        save_path="reports/figures/flower_convergence_macro_f1.png",
        show=True
    )

    # 2. Plot Thesis Section 9 System Metrics (Communication Cost & Parameter Drift)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Left: Cumulative Communication Overhead
    ax1.plot(rounds, round_history["comm_cost_mb"], "o-", color="#2980b9", linewidth=2.2, markersize=6)
    ax1.set_title("Cumulative Network Communication Overhead", fontsize=12, fontweight="bold")
    ax1.set_xlabel("Communication Round (t)", fontsize=11)
    ax1.set_ylabel("Total Transferred (MB)", fontsize=11)
    ax1.grid(True, linestyle="--", alpha=0.6)

    # Right: Client Parameter Drift Norm
    ax2.plot(rounds, round_history["drift_l2"], "s-", color="#e74c3c", linewidth=2.2, markersize=6)
    ax2.set_title("Average Client Parameter Drift ||w_local - w_t||_2", fontsize=12, fontweight="bold")
    ax2.set_xlabel("Communication Round (t)", fontsize=11)
    ax2.set_ylabel("L2 Parameter Drift", fontsize=11)
    ax2.grid(True, linestyle="--", alpha=0.6)

    plt.tight_layout()
    plt.savefig("reports/figures/flower_system_metrics.png", dpi=300, bbox_inches="tight")
    plt.show()

    if round_history["comm_cost_mb"]:
        print(f"Total Cumulative Communication Cost: {round_history['comm_cost_mb'][-1]:.2f} MB")
    if round_history["drift_l2"]:
        print(f"Final Parameter Drift: {round_history['drift_l2'][-1]:.4f}")


---
### 🏆 Step 11: Comprehensive Holdout Test Evaluation
We load the best checkpointed global model and evaluate it on the 20% holdout test set (unseen by all clients):
* **Overall Accuracy & Macro-F1 Score**
* **Per-Class Breakdown (Precision, Recall, F1)**
* **Minority Attack Recall Isolation (`Web-based`, `Brute-force`)**
* **Publication Figures:** Confusion matrix heatmap, per-class bar chart, and minority recall.


In [ ]:
from src.evaluation.evaluator import evaluate_comprehensive
from src.visualization import plot_confusion_matrix, plot_per_class_metrics, plot_minority_recall

# 1. Load best checkpoint weights if available
best_weights_path = os.path.join(strategy.checkpoint_dir, f"flower_{strategy.strategy_name}_best_weights.npz")
if os.path.exists(best_weights_path):
    print(f"Loading best server weights from: {best_weights_path}")
    loaded = np.load(best_weights_path)
    weights = [loaded[f"arr_{i}"] for i in range(len(loaded.files))]
    global_model.set_weights(weights)
else:
    print("Using current global model parameters.")

# 2. Run Comprehensive Evaluation on Holdout Test Set
test_results = evaluate_comprehensive(
    model=global_model,
    dataloader=data["test_loader"],
    class_names=class_names,
    minority_classes=minority_classes,
    device=device
)

print("\n" + "=" * 65)
print(f"🏆 OFFICIAL HOLDOUT TEST BENCHMARK ({CONFIG.experiment_name})")
print("=" * 65)
print(f"Overall Accuracy:        {test_results['accuracy'] * 100:.2f}%")
print(f"Macro-Averaged F1-Score: {test_results['macro_f1'] * 100:.2f}%")
print(f"Weighted F1-Score:       {test_results['weighted_f1'] * 100:.2f}%")
print(f"Macro-Precision:         {test_results['macro_precision'] * 100:.2f}%")
print(f"Macro-Recall:            {test_results['macro_recall'] * 100:.2f}%")
print("-" * 65)
print("Minority Attack Detection Rates (Recall):")
for attack_name, recall_val in test_results["minority_recall"].items():
    print(f"  - {attack_name:12s}: {recall_val * 100:.2f}%")
print("=" * 65)

# 3. Publication Figures
fig_cm = plot_confusion_matrix(
    cm=test_results["confusion_matrix"],
    class_names=class_names,
    title=f"Normalized Confusion Matrix ({CONFIG.experiment_name})",
    save_path="reports/figures/flower_confusion_matrix.png",
    show=True
)

fig_pc = plot_per_class_metrics(
    metrics_dict=test_results,
    class_names=class_names,
    title=f"Per-Class Performance Breakdown ({CONFIG.experiment_name})",
    save_path="reports/figures/flower_per_class_metrics.png",
    show=True
)

if test_results["minority_recall"]:
    fig_min = plot_minority_recall(
        minority_data=test_results["minority_recall"],
        title=f"Stealthy Minority Attack Detection Rates ({CONFIG.experiment_name})",
        save_path="reports/figures/flower_minority_recall.png",
        show=True
    )


---
### 📊 Step 12: Multi-Scenario Benchmark Comparison
We compare this Flower Federated Learning run against the Centralized Upper Bound (Scenario E1) using `plot_scenario_comparison`.


In [ ]:
from src.visualization import plot_scenario_comparison
import json

# Load Centralized E1 result if available
centralized_metrics_file = "reports/centralized_e1_metrics.json"
centralized_f1 = 88.4  # Default benchmark baseline reference
if os.path.exists(centralized_metrics_file):
    with open(centralized_metrics_file) as f:
        c_metrics = json.load(f)
        centralized_f1 = c_metrics.get("macro_f1", 0.884) * 100.0

current_f1 = test_results["macro_f1"] * 100.0

comparison_data = {
    "E1: Centralized Ceiling": centralized_f1,
    f"Flower {strategy.strategy_name.upper()} ({CONFIG.partition_type.upper()})": current_f1
}

fig_comp = plot_scenario_comparison(
    results=comparison_data,
    metric="Macro-F1",
    title="Federated Flower vs Centralized Upper Bound",
    save_path="reports/figures/flower_scenario_comparison.png",
    show=True
)

fl_penalty = centralized_f1 - current_f1
print(f"\nScenario Comparison Summary:")
print(f"  Centralized Upper Bound (E1): {centralized_f1:.2f}%")
print(f"  Flower {strategy.strategy_name.upper()}:                 {current_f1:.2f}%")
print(f"  Federated Non-IID Penalty:    {fl_penalty:+.2f}%")


---
### 🌐 Step 13 (Optional / Deployment): Standalone gRPC Multi-Process Mode
In addition to in-notebook simulation, Flower natively supports distributed multi-machine deployment over standard gRPC network sockets.  
The exact same `FlowerIoTClient` and `FlowerIoTServerStrategy` components can be executed as standalone OS processes:

```bash
# Terminal 1: Launch Flower Central Server (gRPC port 8080)
python -m src.federated.flower_server --server-address 0.0.0.0:8080 --rounds 10 --strategy fedprox --mu 0.05 --min-clients 5

# Terminals 2 to 6: Launch Distributed Edge Clients
python -m src.federated.flower_client --client-id 0 --server-address 127.0.0.1:8080 --device cuda
python -m src.federated.flower_client --client-id 1 --server-address 127.0.0.1:8080 --device cpu
# ...
```


In [ ]:
# 1. Serialize partitions for standalone gRPC processes
from src.data.partitioner import save_partitions_for_grpc

partitions_dir = save_partitions_for_grpc(
    client_partitions=client_partitions,
    X=data["X_train"],
    y=data["y_train"],
    output_dir="checkpoints/partitions",
    X_val=data["X_val"],
    y_val=data["y_val"],
    class_names=class_names
)
print(f"✅ Partitions serialized to: {partitions_dir}")

# 2. Launch Flower gRPC Multi-Process Pipeline (Zero Ray Engine, pure TCP/IP sockets)
# Spawns Server on port 8080 and 5 client processes independently:
!python run_grpc.py --rounds 10 --num-clients 5 --strategy fedprox --mu 0.05 --port 8080 --partitions-dir checkpoints/partitions


---
### 💾 Step 14: Export Checkpoints, Config & Reports to Google Drive
We serialize the best global model weights (`.pth` and `.npz`), preprocessor state (`.pkl`), active hyperparameter config (`.json`), test metrics (`.json`), and round history (`.json`), then permanently archive them and all generated figures directly to `/content/drive/MyDrive/NguyenVietKyQuanKLTN/`.


In [ ]:
import json

# 1. Save artifacts locally
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("reports", exist_ok=True)

model_save_path = f"checkpoints/{CONFIG.experiment_name}_best_model.pth"
global_model.save_weights(model_save_path)

preprocessor_save_path = "checkpoints/flower_preprocessor.pkl"
data["preprocessor"].save(preprocessor_save_path)

config_save_path = f"reports/{CONFIG.experiment_name}_config.json"
CONFIG.save_json(config_save_path)

# Export round history
history_save_path = f"reports/{CONFIG.experiment_name}_round_history.json"
strategy.save_round_history(history_save_path)

# Export test metrics (sanitize numpy arrays)
clean_test_results = {
    k: v for k, v in test_results.items() if k != "confusion_matrix"
}
clean_test_results["confusion_matrix"] = test_results["confusion_matrix"].tolist()
clean_test_results["experiment_name"] = CONFIG.experiment_name
clean_test_results["total_sim_time_sec"] = float(total_sim_time)
clean_test_results["best_round"] = int(strategy.best_round)
clean_test_results["comm_cost_mb"] = float(strategy.cumulative_comm_bytes / (1024**2))

metrics_save_path = f"reports/{CONFIG.experiment_name}_metrics.json"
with open(metrics_save_path, "w") as f:
    json.dump(clean_test_results, f, indent=4)

print("Saved Local Artifacts:")
print(f"  - Model:         {model_save_path}")
print(f"  - Preprocessor:  {preprocessor_save_path}")
print(f"  - Config:        {config_save_path}")
print(f"  - Round History: {history_save_path}")
print(f"  - Test Metrics:  {metrics_save_path}")

# 2. Archive to Google Drive
drive_dest_dir = "/content/drive/MyDrive/NguyenVietKyQuanKLTN"
if os.path.exists("/content/drive/MyDrive"):
    os.makedirs(drive_dest_dir, exist_ok=True)
    drive_figures_dir = os.path.join(drive_dest_dir, "figures")
    os.makedirs(drive_figures_dir, exist_ok=True)
    
    # Copy reports & checkpoints
    shutil.copy(model_save_path, drive_dest_dir)
    shutil.copy(preprocessor_save_path, drive_dest_dir)
    shutil.copy(config_save_path, drive_dest_dir)
    shutil.copy(history_save_path, drive_dest_dir)
    shutil.copy(metrics_save_path, drive_dest_dir)
    
    # Copy figures
    if os.path.exists("reports/figures"):
        for fname in os.listdir("reports/figures"):
            if fname.endswith(".png"):
                shutil.copy(os.path.join("reports/figures", fname), drive_figures_dir)
                
    print(f"\nAll artifacts successfully synchronized to Google Drive: {drive_dest_dir}")
else:
    print("\nGoogle Drive not connected. All artifacts preserved locally in checkpoints/ and reports/.")
